# Walk-Forward Analysis - All 50 Folds
## Aggregate PPO Performance Across Time

**Date:** November 10, 2025  
**Objective:** Evaluate all 50 trained PPO models and aggregate results for robust performance analysis

---

## Contents
1. Setup & Configuration
2. Load All Trained Models
3. Evaluate All Folds (Val & Test)
4. Aggregate Statistics
5. Performance Distribution Analysis
6. Temporal Analysis
7. Risk Metrics Analysis
8. Final Summary & Production Readiness

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# RL imports
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import torch

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Project imports
from harlf.config import TICKERS, get_fold_files
from harlf.envs.portfolio_env import PortfolioEnv
from harlf.agents.dirichlet_policy import SoftmaxActorCriticPolicy

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

N_FOLDS = 50
REWARD_TYPE = 'ema_sharpe'
MODELS_DIR = Path('../models')
SEED = 42

# Set seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"📊 Configuration")
print("="*70)
print(f"Total folds: {N_FOLDS}")
print(f"Reward type: {REWARD_TYPE}")
print(f"Models directory: {MODELS_DIR.absolute()}")
print(f"Seed: {SEED}")

## 2. Helper Functions

In [ ]:
def make_env(fold_id, split, reward_type='ema_sharpe'):
    """Create a portfolio environment for a given fold and split."""
    env = PortfolioEnv(fold_id=fold_id, split=split, reward_type=reward_type)
    env = Monitor(env)
    return env


def evaluate_agent(agent, env, fold_id, split, deterministic=True):
    """
    Evaluate an agent on an environment.
    
    Returns:
        Dictionary with performance metrics
    """
    obs, info = env.reset()
    done = False
    truncated = False
    
    episode_returns = []
    episode_weights = []
    episode_values = [info['portfolio_value']]
    
    while not (done or truncated):
        action, _states = agent.predict(obs, deterministic=deterministic)
        obs, reward, done, truncated, info = env.step(action)
        
        episode_returns.append(info['portfolio_return'])
        episode_weights.append(info['weights'])
        episode_values.append(info['portfolio_value'])
    
    # Get episode metrics from unwrapped env
    unwrapped_env = env.unwrapped if hasattr(env, 'unwrapped') else env
    if hasattr(unwrapped_env, 'get_episode_metrics'):
        metrics = unwrapped_env.get_episode_metrics()
    else:
        metrics = {}
    
    # Add fold and split info
    metrics['fold_id'] = fold_id
    metrics['split'] = split
    
    return metrics


class BaselineStrategy:
    """Base class for baseline strategies."""
    
    def __init__(self, n_assets=7):
        self.n_assets = n_assets


class EqualWeightStrategy(BaselineStrategy):
    """Equal weight (1/N) strategy."""
    
    def __init__(self, n_assets=7):
        super().__init__(n_assets)
        self.weights = np.ones(n_assets) / n_assets
    
    def predict(self, observation, deterministic=True):
        return self.weights, None


print("✅ Helper functions defined")

## 3. Check Model Availability

In [ ]:
# Check which models exist
print("\n🔍 Checking for trained models...\n")

available_models = []
missing_models = []

for fold_id in range(N_FOLDS):
    model_path = MODELS_DIR / f'fold_{fold_id}' / f'ppo_fold_{fold_id}_final.zip'
    if model_path.exists():
        available_models.append(fold_id)
    else:
        missing_models.append(fold_id)

print(f"📊 Model Availability:")
print("="*70)
print(f"✅ Available models: {len(available_models)}/{N_FOLDS}")
print(f"❌ Missing models: {len(missing_models)}/{N_FOLDS}")

if len(available_models) > 0:
    print(f"\nAvailable folds: {available_models[:10]}{'...' if len(available_models) > 10 else ''}")

if len(missing_models) > 0:
    print(f"\n⚠️  Missing folds: {missing_models[:10]}{'...' if len(missing_models) > 10 else ''}")
    print(f"   Run notebook 02_train_base_agents_all_folds.ipynb to train missing models")

if len(available_models) == 0:
    raise ValueError("No trained models found! Please train models first.")

# Use only available models for analysis
FOLDS_TO_EVALUATE = available_models
print(f"\n📈 Will evaluate {len(FOLDS_TO_EVALUATE)} folds")

## 4. Evaluate All Folds

Load each model and evaluate on validation and test sets.

In [ ]:
# Evaluate all available models
print("\n" + "="*70)
print("🔍 EVALUATING ALL MODELS")
print("="*70)

ppo_results_val = []
ppo_results_test = []
baseline_results_val = []
baseline_results_test = []

for fold_id in tqdm(FOLDS_TO_EVALUATE, desc="Evaluating folds"):
    try:
        # Load model
        model_path = MODELS_DIR / f'fold_{fold_id}' / f'ppo_fold_{fold_id}_final.zip'
        model = PPO.load(str(model_path))
        
        # Create environments
        val_env = make_env(fold_id, 'val', REWARD_TYPE)
        test_env = make_env(fold_id, 'test', REWARD_TYPE)
        
        # Evaluate PPO
        val_metrics = evaluate_agent(model, val_env, fold_id, 'val', deterministic=True)
        test_metrics = evaluate_agent(model, test_env, fold_id, 'test', deterministic=True)
        
        ppo_results_val.append(val_metrics)
        ppo_results_test.append(test_metrics)
        
        # Evaluate baseline (Equal Weight) - only once per fold
        if fold_id == FOLDS_TO_EVALUATE[0] or fold_id % 10 == 0:  # Evaluate baseline every 10 folds
            baseline = EqualWeightStrategy()
            baseline_val_metrics = evaluate_agent(baseline, val_env, fold_id, 'val', deterministic=True)
            baseline_test_metrics = evaluate_agent(baseline, test_env, fold_id, 'test', deterministic=True)
            baseline_results_val.append(baseline_val_metrics)
            baseline_results_test.append(baseline_test_metrics)
        
        # Clean up
        val_env.close()
        test_env.close()
        del model, val_env, test_env
        
    except Exception as e:
        print(f"\n❌ Error evaluating fold {fold_id}: {e}")
        continue

print(f"\n✅ Evaluation complete!")
print(f"   PPO Val results: {len(ppo_results_val)}")
print(f"   PPO Test results: {len(ppo_results_test)}")
print(f"   Baseline Val results: {len(baseline_results_val)}")
print(f"   Baseline Test results: {len(baseline_results_test)}")

## 5. Create Results DataFrames

In [ ]:
# Convert to DataFrames
ppo_val_df = pd.DataFrame(ppo_results_val)
ppo_test_df = pd.DataFrame(ppo_results_test)
baseline_val_df = pd.DataFrame(baseline_results_val)
baseline_test_df = pd.DataFrame(baseline_results_test)

print("📊 Results DataFrames created")
print(f"   PPO Validation: {ppo_val_df.shape}")
print(f"   PPO Test: {ppo_test_df.shape}")
print(f"\nSample PPO validation results:")
display(ppo_val_df.head())

## 6. Aggregate Statistics

In [ ]:
# Define key metrics to analyze
key_metrics = ['sharpe_ratio', 'total_return', 'volatility', 'max_drawdown', 
               'sortino_ratio', 'mean_turnover', 'episode_length']

# Compute aggregate statistics
def compute_aggregate_stats(df, metrics):
    """Compute mean, std, median, min, max for each metric."""
    stats = []
    for metric in metrics:
        if metric in df.columns:
            stats.append({
                'Metric': metric,
                'Mean': df[metric].mean(),
                'Std': df[metric].std(),
                'Median': df[metric].median(),
                'Min': df[metric].min(),
                'Max': df[metric].max(),
                '25th': df[metric].quantile(0.25),
                '75th': df[metric].quantile(0.75),
            })
    return pd.DataFrame(stats)

# PPO Validation Stats
ppo_val_stats = compute_aggregate_stats(ppo_val_df, key_metrics)

print("\n" + "="*100)
print("📊 PPO VALIDATION SET - AGGREGATE STATISTICS (ALL FOLDS)")
print("="*100)
display(ppo_val_stats.style.format({
    'Mean': '{:.4f}',
    'Std': '{:.4f}',
    'Median': '{:.4f}',
    'Min': '{:.4f}',
    'Max': '{:.4f}',
    '25th': '{:.4f}',
    '75th': '{:.4f}',
}))

# PPO Test Stats
ppo_test_stats = compute_aggregate_stats(ppo_test_df, key_metrics)

print("\n" + "="*100)
print("📊 PPO TEST SET - AGGREGATE STATISTICS (ALL FOLDS)")
print("="*100)
display(ppo_test_stats.style.format({
    'Mean': '{:.4f}',
    'Std': '{:.4f}',
    'Median': '{:.4f}',
    'Min': '{:.4f}',
    'Max': '{:.4f}',
    '25th': '{:.4f}',
    '75th': '{:.4f}',
}))

## 7. Compare PPO vs Baseline

In [ ]:
# Compare key metrics
if len(baseline_val_df) > 0:
    comparison_metrics = ['sharpe_ratio', 'total_return', 'max_drawdown', 'volatility']
    
    comparison_data = []
    for metric in comparison_metrics:
        if metric in ppo_val_df.columns and metric in baseline_val_df.columns:
            comparison_data.append({
                'Metric': metric,
                'PPO Mean': ppo_val_df[metric].mean(),
                'Baseline Mean': baseline_val_df[metric].mean(),
                'Difference': ppo_val_df[metric].mean() - baseline_val_df[metric].mean(),
                'PPO Better': ppo_val_df[metric].mean() > baseline_val_df[metric].mean() if metric != 'max_drawdown' else ppo_val_df[metric].mean() > baseline_val_df[metric].mean(),
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n" + "="*100)
    print("⚖️  PPO vs EQUAL WEIGHT BASELINE - VALIDATION SET")
    print("="*100)
    display(comparison_df.style.format({
        'PPO Mean': '{:.4f}',
        'Baseline Mean': '{:.4f}',
        'Difference': '{:.4f}',
    }))
else:
    print("⚠️  No baseline results available for comparison")

## 8. Performance Distribution Visualizations

In [ ]:
# Plot 1: Distribution of key metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics_to_plot = ['sharpe_ratio', 'total_return', 'volatility', 'max_drawdown', 'sortino_ratio', 'mean_turnover']
metric_labels = ['Sharpe Ratio', 'Total Return', 'Volatility', 'Max Drawdown', 'Sortino Ratio', 'Avg Turnover']

for i, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    if metric in ppo_val_df.columns:
        # Validation
        axes[i].hist(ppo_val_df[metric], bins=20, alpha=0.6, label='Validation', color='steelblue', edgecolor='black')
        # Test
        if metric in ppo_test_df.columns:
            axes[i].hist(ppo_test_df[metric], bins=20, alpha=0.6, label='Test', color='orange', edgecolor='black')
        
        # Add mean lines
        val_mean = ppo_val_df[metric].mean()
        axes[i].axvline(val_mean, color='steelblue', linestyle='--', linewidth=2, label=f'Val Mean: {val_mean:.3f}')
        
        if metric in ppo_test_df.columns:
            test_mean = ppo_test_df[metric].mean()
            axes[i].axvline(test_mean, color='orange', linestyle='--', linewidth=2, label=f'Test Mean: {test_mean:.3f}')
        
        axes[i].set_title(label, fontsize=12, fontweight='bold')
        axes[i].set_xlabel(label)
        axes[i].set_ylabel('Frequency')
        axes[i].legend(fontsize=8)
        axes[i].grid(alpha=0.3)

plt.suptitle(f'PPO Performance Distribution Across {len(FOLDS_TO_EVALUATE)} Folds', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Box plots for key metrics
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

box_metrics = ['sharpe_ratio', 'total_return', 'max_drawdown', 'volatility']
box_labels = ['Sharpe Ratio', 'Total Return', 'Max Drawdown', 'Volatility']

for i, (metric, label) in enumerate(zip(box_metrics, box_labels)):
    if metric in ppo_val_df.columns:
        data_to_plot = [ppo_val_df[metric].dropna()]
        labels = ['Validation']
        
        if metric in ppo_test_df.columns:
            data_to_plot.append(ppo_test_df[metric].dropna())
            labels.append('Test')
        
        bp = axes[i].boxplot(data_to_plot, labels=labels, patch_artist=True,
                             boxprops=dict(facecolor='lightblue', alpha=0.7),
                             medianprops=dict(color='red', linewidth=2),
                             whiskerprops=dict(linewidth=1.5),
                             capprops=dict(linewidth=1.5))
        
        axes[i].set_title(label, fontsize=12, fontweight='bold')
        axes[i].set_ylabel(label)
        axes[i].grid(axis='y', alpha=0.3)
        
        # Add mean markers
        for j, data in enumerate(data_to_plot, 1):
            mean_val = data.mean()
            axes[i].plot(j, mean_val, 'D', color='green', markersize=8, label='Mean' if j == 1 else '')
        
        if i == 0:
            axes[i].legend()

plt.suptitle(f'PPO Performance Box Plots Across {len(FOLDS_TO_EVALUATE)} Folds', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Temporal Analysis

Analyze how performance varies across time (folds).

In [ ]:
# Plot performance over folds
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

temporal_metrics = ['sharpe_ratio', 'total_return', 'max_drawdown', 'volatility']
temporal_labels = ['Sharpe Ratio', 'Total Return', 'Max Drawdown', 'Volatility']

for i, (metric, label) in enumerate(zip(temporal_metrics, temporal_labels)):
    if metric in ppo_val_df.columns:
        # Validation over folds
        axes[i].plot(ppo_val_df['fold_id'], ppo_val_df[metric], 
                    'o-', alpha=0.6, label='Validation', color='steelblue', markersize=4)
        
        # Test over folds
        if metric in ppo_test_df.columns:
            axes[i].plot(ppo_test_df['fold_id'], ppo_test_df[metric], 
                        's-', alpha=0.6, label='Test', color='orange', markersize=4)
        
        # Add mean lines
        val_mean = ppo_val_df[metric].mean()
        axes[i].axhline(val_mean, color='steelblue', linestyle='--', linewidth=2, alpha=0.7)
        
        if metric in ppo_test_df.columns:
            test_mean = ppo_test_df[metric].mean()
            axes[i].axhline(test_mean, color='orange', linestyle='--', linewidth=2, alpha=0.7)
        
        axes[i].set_title(f'{label} Over Time', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Fold ID')
        axes[i].set_ylabel(label)
        axes[i].legend()
        axes[i].grid(alpha=0.3)

plt.suptitle(f'PPO Performance Over Time ({len(FOLDS_TO_EVALUATE)} Folds)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Rolling statistics (if enough folds)
if len(FOLDS_TO_EVALUATE) >= 10:
    window = 10
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    
    # Sharpe ratio rolling mean and std
    if 'sharpe_ratio' in ppo_val_df.columns:
        ppo_val_sorted = ppo_val_df.sort_values('fold_id')
        rolling_mean = ppo_val_sorted['sharpe_ratio'].rolling(window=window).mean()
        rolling_std = ppo_val_sorted['sharpe_ratio'].rolling(window=window).std()
        
        axes[0].plot(ppo_val_sorted['fold_id'], ppo_val_sorted['sharpe_ratio'], 
                    'o', alpha=0.3, color='gray', label='Raw')
        axes[0].plot(ppo_val_sorted['fold_id'], rolling_mean, 
                    '-', linewidth=2, color='steelblue', label=f'{window}-Fold Rolling Mean')
        axes[0].fill_between(ppo_val_sorted['fold_id'], 
                            rolling_mean - rolling_std, 
                            rolling_mean + rolling_std, 
                            alpha=0.3, color='steelblue', label='±1 Std')
        
        axes[0].set_title(f'Sharpe Ratio - Rolling Statistics (Window={window})', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Fold ID')
        axes[0].set_ylabel('Sharpe Ratio')
        axes[0].legend()
        axes[0].grid(alpha=0.3)
    
    # Total return rolling mean and std
    if 'total_return' in ppo_val_df.columns:
        rolling_mean = ppo_val_sorted['total_return'].rolling(window=window).mean()
        rolling_std = ppo_val_sorted['total_return'].rolling(window=window).std()
        
        axes[1].plot(ppo_val_sorted['fold_id'], ppo_val_sorted['total_return'], 
                    'o', alpha=0.3, color='gray', label='Raw')
        axes[1].plot(ppo_val_sorted['fold_id'], rolling_mean, 
                    '-', linewidth=2, color='green', label=f'{window}-Fold Rolling Mean')
        axes[1].fill_between(ppo_val_sorted['fold_id'], 
                            rolling_mean - rolling_std, 
                            rolling_mean + rolling_std, 
                            alpha=0.3, color='green', label='±1 Std')
        
        axes[1].set_title(f'Total Return - Rolling Statistics (Window={window})', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Fold ID')
        axes[1].set_ylabel('Total Return')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Not enough folds for rolling statistics (need at least 10)")

## 10. Risk Analysis

In [ ]:
# Risk-Return scatter for all folds
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Validation set
if 'volatility' in ppo_val_df.columns and 'total_return' in ppo_val_df.columns:
    # Compute annualized return
    ppo_val_df['ann_return'] = ppo_val_df['total_return'] * (252 / ppo_val_df['episode_length'])
    
    axes[0].scatter(ppo_val_df['volatility'], ppo_val_df['ann_return'], 
                   alpha=0.6, s=50, c=ppo_val_df['sharpe_ratio'], 
                   cmap='RdYlGn', edgecolors='black', linewidth=0.5)
    
    # Add colorbar
    cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
    cbar.set_label('Sharpe Ratio', rotation=270, labelpad=20)
    
    axes[0].set_title('Risk-Return Profile - Validation Set', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Volatility (Annualized)')
    axes[0].set_ylabel('Return (Annualized)')
    axes[0].grid(alpha=0.3)
    axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Test set
if 'volatility' in ppo_test_df.columns and 'total_return' in ppo_test_df.columns:
    # Compute annualized return
    ppo_test_df['ann_return'] = ppo_test_df['total_return'] * (252 / ppo_test_df['episode_length'])
    
    axes[1].scatter(ppo_test_df['volatility'], ppo_test_df['ann_return'], 
                   alpha=0.6, s=50, c=ppo_test_df['sharpe_ratio'], 
                   cmap='RdYlGn', edgecolors='black', linewidth=0.5)
    
    # Add colorbar
    cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
    cbar.set_label('Sharpe Ratio', rotation=270, labelpad=20)
    
    axes[1].set_title('Risk-Return Profile - Test Set', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Volatility (Annualized)')
    axes[1].set_ylabel('Return (Annualized)')
    axes[1].grid(alpha=0.3)
    axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.suptitle(f'Risk-Return Analysis Across {len(FOLDS_TO_EVALUATE)} Folds', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown analysis
if 'max_drawdown' in ppo_val_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    
    # Histogram of max drawdowns
    axes[0].hist(ppo_val_df['max_drawdown'], bins=20, alpha=0.7, 
                color='red', edgecolor='black', label='Validation')
    if 'max_drawdown' in ppo_test_df.columns:
        axes[0].hist(ppo_test_df['max_drawdown'], bins=20, alpha=0.7, 
                    color='orange', edgecolor='black', label='Test')
    
    axes[0].axvline(ppo_val_df['max_drawdown'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f"Val Mean: {ppo_val_df['max_drawdown'].mean():.3f}")
    axes[0].axvline(-0.2, color='black', linestyle='--', linewidth=2, 
                   label='Episode termination threshold', alpha=0.5)
    
    axes[0].set_title('Max Drawdown Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Max Drawdown')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Drawdown vs Sharpe
    if 'sharpe_ratio' in ppo_val_df.columns:
        axes[1].scatter(ppo_val_df['max_drawdown'], ppo_val_df['sharpe_ratio'], 
                       alpha=0.6, s=50, color='steelblue', edgecolors='black', 
                       linewidth=0.5, label='Validation')
        
        if 'sharpe_ratio' in ppo_test_df.columns and 'max_drawdown' in ppo_test_df.columns:
            axes[1].scatter(ppo_test_df['max_drawdown'], ppo_test_df['sharpe_ratio'], 
                           alpha=0.6, s=50, color='orange', edgecolors='black', 
                           linewidth=0.5, label='Test')
        
        axes[1].set_title('Max Drawdown vs Sharpe Ratio', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Max Drawdown')
        axes[1].set_ylabel('Sharpe Ratio')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        axes[1].axvline(-0.2, color='black', linestyle='--', linewidth=2, alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Count folds that hit max drawdown
    hit_dd_val = (ppo_val_df['max_drawdown'] <= -0.2).sum()
    hit_dd_test = (ppo_test_df['max_drawdown'] <= -0.2).sum() if 'max_drawdown' in ppo_test_df.columns else 0
    
    print(f"\n⚠️  Folds that hit max drawdown threshold (-20%):")
    print(f"   Validation: {hit_dd_val}/{len(ppo_val_df)} ({hit_dd_val/len(ppo_val_df)*100:.1f}%)")
    print(f"   Test: {hit_dd_test}/{len(ppo_test_df)} ({hit_dd_test/len(ppo_test_df)*100:.1f}%)")

## 11. Consistency Analysis

In [ ]:
# Check consistency metrics
print("\n" + "="*70)
print("📊 CONSISTENCY ANALYSIS")
print("="*70)

# Sharpe ratio consistency
if 'sharpe_ratio' in ppo_val_df.columns:
    positive_sharpe_val = (ppo_val_df['sharpe_ratio'] > 0).sum()
    positive_sharpe_test = (ppo_test_df['sharpe_ratio'] > 0).sum() if 'sharpe_ratio' in ppo_test_df.columns else 0
    
    print(f"\n✅ Positive Sharpe Ratio:")
    print(f"   Validation: {positive_sharpe_val}/{len(ppo_val_df)} ({positive_sharpe_val/len(ppo_val_df)*100:.1f}%)")
    print(f"   Test: {positive_sharpe_test}/{len(ppo_test_df)} ({positive_sharpe_test/len(ppo_test_df)*100:.1f}%)")

# Return consistency
if 'total_return' in ppo_val_df.columns:
    positive_return_val = (ppo_val_df['total_return'] > 0).sum()
    positive_return_test = (ppo_test_df['total_return'] > 0).sum() if 'total_return' in ppo_test_df.columns else 0
    
    print(f"\n💰 Positive Returns:")
    print(f"   Validation: {positive_return_val}/{len(ppo_val_df)} ({positive_return_val/len(ppo_val_df)*100:.1f}%)")
    print(f"   Test: {positive_return_test}/{len(ppo_test_df)} ({positive_return_test/len(ppo_test_df)*100:.1f}%)")

# Coefficient of variation (std/mean) - lower is more consistent
if 'sharpe_ratio' in ppo_val_df.columns:
    cv_sharpe_val = ppo_val_df['sharpe_ratio'].std() / abs(ppo_val_df['sharpe_ratio'].mean())
    cv_sharpe_test = ppo_test_df['sharpe_ratio'].std() / abs(ppo_test_df['sharpe_ratio'].mean()) if 'sharpe_ratio' in ppo_test_df.columns else 0
    
    print(f"\n📈 Coefficient of Variation (Sharpe Ratio):")
    print(f"   Validation: {cv_sharpe_val:.4f} (lower = more consistent)")
    print(f"   Test: {cv_sharpe_test:.4f}")

# Episode length consistency
if 'episode_length' in ppo_val_df.columns:
    avg_length_val = ppo_val_df['episode_length'].mean()
    avg_length_test = ppo_test_df['episode_length'].mean() if 'episode_length' in ppo_test_df.columns else 0
    max_possible = 21  # Assuming 21 days per episode
    
    print(f"\n📏 Episode Length (days):")
    print(f"   Validation: {avg_length_val:.1f}/{max_possible} ({avg_length_val/max_possible*100:.1f}% of max)")
    print(f"   Test: {avg_length_test:.1f}/{max_possible} ({avg_length_test/max_possible*100:.1f}% of max)")

## 12. Final Summary & Production Readiness

In [ ]:
print("\n" + "="*70)
print("🎯 FINAL SUMMARY - WALK-FORWARD VALIDATION")
print("="*70)

print(f"\n📊 Dataset:")
print(f"   Total folds evaluated: {len(FOLDS_TO_EVALUATE)}/{N_FOLDS}")
print(f"   Validation episodes: {len(ppo_val_df)}")
print(f"   Test episodes: {len(ppo_test_df)}")

if 'sharpe_ratio' in ppo_val_df.columns and 'total_return' in ppo_val_df.columns:
    print(f"\n🏆 Key Performance Metrics (Validation):")
    print(f"   Mean Sharpe Ratio: {ppo_val_df['sharpe_ratio'].mean():.4f} ± {ppo_val_df['sharpe_ratio'].std():.4f}")
    print(f"   Median Sharpe Ratio: {ppo_val_df['sharpe_ratio'].median():.4f}")
    print(f"   Mean Total Return: {ppo_val_df['total_return'].mean():.4f} ± {ppo_val_df['total_return'].std():.4f}")
    print(f"   Median Total Return: {ppo_val_df['total_return'].median():.4f}")
    
    if 'max_drawdown' in ppo_val_df.columns:
        print(f"   Mean Max Drawdown: {ppo_val_df['max_drawdown'].mean():.4f} ± {ppo_val_df['max_drawdown'].std():.4f}")
    
    if 'volatility' in ppo_val_df.columns:
        print(f"   Mean Volatility: {ppo_val_df['volatility'].mean():.4f} ± {ppo_val_df['volatility'].std():.4f}")

if 'sharpe_ratio' in ppo_test_df.columns and 'total_return' in ppo_test_df.columns:
    print(f"\n🎯 Key Performance Metrics (Test):")
    print(f"   Mean Sharpe Ratio: {ppo_test_df['sharpe_ratio'].mean():.4f} ± {ppo_test_df['sharpe_ratio'].std():.4f}")
    print(f"   Median Sharpe Ratio: {ppo_test_df['sharpe_ratio'].median():.4f}")
    print(f"   Mean Total Return: {ppo_test_df['total_return'].mean():.4f} ± {ppo_test_df['total_return'].std():.4f}")
    print(f"   Median Total Return: {ppo_test_df['total_return'].median():.4f}")

# Production readiness checklist
print(f"\n{'='*70}")
print("🚀 PRODUCTION READINESS CHECKLIST")
print(f"{'='*70}")

checklist = []

# Check 1: Sufficient folds
if len(FOLDS_TO_EVALUATE) >= 40:
    checklist.append(("✅", f"Sufficient folds evaluated ({len(FOLDS_TO_EVALUATE)}/50)"))
elif len(FOLDS_TO_EVALUATE) >= 20:
    checklist.append(("⚠️ ", f"Moderate folds evaluated ({len(FOLDS_TO_EVALUATE)}/50) - consider more"))
else:
    checklist.append(("❌", f"Insufficient folds ({len(FOLDS_TO_EVALUATE)}/50) - need more data"))

# Check 2: Positive Sharpe ratio
if 'sharpe_ratio' in ppo_test_df.columns:
    mean_sharpe = ppo_test_df['sharpe_ratio'].mean()
    if mean_sharpe > 1.0:
        checklist.append(("✅", f"Strong Sharpe ratio (test mean: {mean_sharpe:.3f})"))
    elif mean_sharpe > 0.5:
        checklist.append(("⚠️ ", f"Moderate Sharpe ratio (test mean: {mean_sharpe:.3f})"))
    else:
        checklist.append(("❌", f"Low Sharpe ratio (test mean: {mean_sharpe:.3f})"))

# Check 3: Consistency
if 'sharpe_ratio' in ppo_test_df.columns:
    positive_rate = (ppo_test_df['sharpe_ratio'] > 0).mean()
    if positive_rate > 0.7:
        checklist.append(("✅", f"High consistency ({positive_rate*100:.1f}% positive Sharpe)"))
    elif positive_rate > 0.5:
        checklist.append(("⚠️ ", f"Moderate consistency ({positive_rate*100:.1f}% positive Sharpe)"))
    else:
        checklist.append(("❌", f"Low consistency ({positive_rate*100:.1f}% positive Sharpe)"))

# Check 4: Positive returns
if 'total_return' in ppo_test_df.columns:
    mean_return = ppo_test_df['total_return'].mean()
    if mean_return > 0.01:
        checklist.append(("✅", f"Positive returns (test mean: {mean_return:.4f})"))
    elif mean_return > 0:
        checklist.append(("⚠️ ", f"Marginally positive returns (test mean: {mean_return:.4f})"))
    else:
        checklist.append(("❌", f"Negative returns (test mean: {mean_return:.4f})"))

# Check 5: Episode completion
if 'episode_length' in ppo_test_df.columns:
    avg_length = ppo_test_df['episode_length'].mean()
    max_length = 21  # Assuming 21 days
    completion_rate = avg_length / max_length
    if completion_rate > 0.9:
        checklist.append(("✅", f"Episodes rarely terminated early ({completion_rate*100:.1f}% complete)"))
    elif completion_rate > 0.7:
        checklist.append(("⚠️ ", f"Some early terminations ({completion_rate*100:.1f}% complete)"))
    else:
        checklist.append(("❌", f"Frequent early terminations ({completion_rate*100:.1f}% complete)"))

# Print checklist
for status, message in checklist:
    print(f"{status} {message}")

# Overall assessment
n_pass = sum(1 for s, _ in checklist if s == "✅")
n_warn = sum(1 for s, _ in checklist if s == "⚠️ ")
n_fail = sum(1 for s, _ in checklist if s == "❌")

print(f"\n{'='*70}")
if n_fail == 0 and n_warn <= 1:
    print("🎉 READY FOR PRODUCTION")
    print("   Model demonstrates strong, consistent performance across folds")
elif n_fail <= 1:
    print("⚠️  NEEDS IMPROVEMENT")
    print("   Model shows promise but requires optimization")
else:
    print("❌ NOT READY FOR PRODUCTION")
    print("   Significant improvements needed before deployment")
print(f"{'='*70}")

print(f"\n📝 Recommendation:")
if n_fail > 1:
    print("   1. Increase training timesteps (500k-1M)")
    print("   2. Tune hyperparameters (learning rate, entropy coefficient)")
    print("   3. Adjust risk management (max drawdown threshold)")
    print("   4. Consider ensemble approaches")
elif n_warn > 1:
    print("   1. Fine-tune hyperparameters for better consistency")
    print("   2. Analyze worst-performing folds for patterns")
    print("   3. Consider position sizing adjustments")
else:
    print("   1. Proceed with out-of-sample testing")
    print("   2. Implement in paper trading environment")
    print("   3. Monitor performance continuously")
    print("   4. Set up automated retraining pipeline")

print("\n" + "="*70)
print("✅ Walk-Forward Analysis Complete")
print("="*70)

## 13. Export Results

Save aggregate results for further analysis or reporting.

In [ ]:
# Export results to CSV
results_dir = Path('../results')
results_dir.mkdir(parents=True, exist_ok=True)

# Save validation results
ppo_val_df.to_csv(results_dir / 'ppo_validation_all_folds.csv', index=False)
print(f"✅ Saved validation results to {results_dir / 'ppo_validation_all_folds.csv'}")

# Save test results
ppo_test_df.to_csv(results_dir / 'ppo_test_all_folds.csv', index=False)
print(f"✅ Saved test results to {results_dir / 'ppo_test_all_folds.csv'}")

# Save aggregate statistics
ppo_val_stats.to_csv(results_dir / 'ppo_validation_aggregate_stats.csv', index=False)
print(f"✅ Saved aggregate statistics to {results_dir / 'ppo_validation_aggregate_stats.csv'}")

ppo_test_stats.to_csv(results_dir / 'ppo_test_aggregate_stats.csv', index=False)
print(f"✅ Saved test aggregate statistics to {results_dir / 'ppo_test_aggregate_stats.csv'}")

print(f"\n📁 All results exported to: {results_dir.absolute()}")

---

## Notebook Complete ✅

This notebook provides comprehensive walk-forward validation analysis across all trained folds.

### Next Steps:
1. Review aggregate statistics and identify improvement areas
2. Analyze worst-performing folds for patterns
3. Consider hyperparameter optimization
4. If results are satisfactory, proceed to hierarchical agent training
5. Set up production monitoring and retraining pipeline